# Chapter 16 — Capstone: Governed Banking Complaint Agent

*Every prior chapter, wired into one system.*

## Objective

Assemble the complaint agent. Run it against the 20 synthetic eval cases. Walk through the audit log for three representative cases — a routine inquiry, an adversarial UDAAP complaint (an overdraft fee alleged to be unfair) and a PII-laden message — and produce a summary report.

In [ ]:
import json
from pathlib import Path

from forgeloop.agents.capstone import build_complaint_harness
from forgeloop.agents.core import Budget, BudgetTracker, TaskSpec

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
policies_dir = root / 'data' / 'policies'
cases_path   = root / 'data' / 'eval_cases' / 'cases.json'
cases = json.loads(cases_path.read_text())
print(f'cases: {len(cases)}  policies: {sorted(p.name for p in policies_dir.glob("*.txt"))}')

## Build the harness

`build_complaint_harness` wires the agent, the five domain tools and the input policies (PII, prompt injection, prohibited advice, fee waiver) into a single `GovernanceHarness`.

All five tools are model-backed by an open-weight Qwen — a shared Qwen3-4B-Instruct for the classifier, draft writer and regulatory proposer, and the larger Qwen3-4B-Instruct for policy synthesis — each paired with a deterministic backstop so a regulated decision never rests on an unverifiable inference; each artifact loads on first use and caches for the process. `flag_regulatory`'s backstop is the richest: a separately trained, calibrated GMS store that verifies and corrects the model's proposed flags, *reads the message geometrically* through a fine-tuned embedding to catch the flags a keyword would miss, and decides escalation by walking the graph.

| Tool | Backed by | Artifact |
| --- | --- | --- |
| `classify_complaint` | Qwen encoder + LoRA adapter + classification head | `data/complaint_classifier_qwen/` |
| `extract_facts`      | GEODE parse-and-bind on the policy store (grounded) + Qwen/rules fallback | `data/gms_policy_store_cap/` |
| `search_policy`      | operator-native GEODE RAG (reuses the extraction) + calibrated head-bind floor + Qwen3-4B synthesis + self-verification | `data/gms_policy_store_cap/` |
| `flag_regulatory`    | grounds on the bound facts + Qwen proposes + GMS guard verifies/corrects + geometric reader | `data/gms_regulatory_store/`, `data/gms_regulatory_cap/` |
| `draft_response`     | Qwen + LoRA (complaints), template (inquiry / other) | `data/draft_response_lm_qwen/` |

In [ ]:
harness, registry = build_complaint_harness(policies_dir=policies_dir)
print('registered tools:', [t.name for t in registry.all()])

## Run a routine inquiry

In [ ]:
def did_escalate(traj):
    if traj.final_state.status in ('escalated', 'failed'): return True
    out = traj.final_state.final_output or {}
    return isinstance(out, dict) and out.get('recommended_action') == 'escalate'

case = next(c for c in cases if c['id'] == 'case-002')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
print('message:        ', case['message'])
print('status:         ', traj.final_state.status)
print('final output:   ', traj.final_state.final_output)

## Run an adversarial overdraft case — must escalate via UDAAP

In [ ]:
case = next(c for c in cases if c['id'] == 'case-016')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
print('message:    ', case['message'])
print('status:     ', traj.final_state.status)
esc = next(r for r in traj.records if r.action.kind == 'escalate')
print('escalation: ', esc.action.reason)

## Visualizing the escalation

The escalation above is recorded in `traj`. We can draw it straight from that trajectory — no re-run of the agent. `kg_process.process_figure` shows the workflow the agent executed, each step annotated with its **recorded** output; `severity_subgraph` draws the stored regulatory triplets `flag_regulatory` named in its `severity_paths`, on the embedding sphere. (`kg_sphere.py` and `kg_process.py` sit beside this notebook.)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # kg_sphere.py / kg_process.py sit beside this notebook
import kg_sphere as K
import kg_process as P

records = P.records_from_trajectory(traj)   # traj = the case-016 run above, reused as-is
fig = P.process_figure(records, title='case-016 · agent trajectory (recorded)')
fig

In [ ]:
# The stored regulatory triplets flag_regulatory walked, on the embedding sphere:
# the evidence that supported the flag (has_evidence) and the severity walk.
_reg_store = str(root / 'data' / 'gms_regulatory_store')
kg_reg = K.load_gms_store(_reg_store, source='v')
sub = P.escalation_subgraph(kg_reg, records, store_path=_reg_store)
print(f'{len(sub.labels)} entities, {len(sub.triples)} triples')
for h, r, t in sub.triples:
    print(f'  {h}  {r}  {t}')
if sub.labels:
    K.visualize(sub, title='case-016 · UDAAP escalation path', arrow_size=0.10)
else:
    print('flag_regulatory was denied by gms_plausibility before it could fire — '
          'no severity path was recorded.\n'
          'To see the UDAAP sphere: delete data/gms_banking_store/calibration.json '
          'and rerun the Tier-1 calibration stage in 00_setup.')

## Why the graph, not dense retrieval

Escalating this complaint needs a multi-hop chain: the message's evidence → the regulation → its severity → the `escalate` action. The graph walked it above. A conventional dense-embedding RAG retrieves by similarity to the query in one shot, with no traversal. Below, the book's dense foil (`agentlab.capstone.dense_rag.DenseRagRetriever`, frozen all-MiniLM-L6-v2) ranks the sections of the same regulatory reference against the complaint. The section that carries the escalate decision (§4 Severity and Escalation) is written in the vocabulary of severities and dispositions — words the complaint never uses — so it ranks near the bottom and falls outside the retriever's top-k. Dense retrieval cannot reach the bridge entity (`UDAAP`, `high`) it never saw; the graph reaches it by relation edges.

In [ ]:
from forgeloop.agents.capstone.dense_rag import DenseRagRetriever

msg = case['message']   # the case-016 complaint, still in scope from the run above

doc = root / 'data' / 'gms_regulatory_store' / 'documents' / 'combined.md'
dense = DenseRagRetriever(doc_path=doc, chunking='section')
hits = dense._retrieve(msg, k=len(dense.chunks))   # existing method: [(chunk_idx, cosine)] ranked

def _title(i):
    return next((l.strip() for l in dense.chunks[i].splitlines() if l.strip()), '')[:52]

print('dense cosine ranking of the regulatory reference vs the complaint:')
for rank, (i, sc) in enumerate(hits):
    mark = '   <- carries the escalate rule' if 'severity and escalation' in _title(i).lower() else ''
    print(f'  #{rank + 1}  {sc:.3f}  {_title(i)}{mark}')

top = [_title(i) for i, _ in hits[:dense.top_k]]
print(f'\ndense retrieves top-{dense.top_k}: {top}')

_flag_out = next(
    (r['output'] for r in records if r.get('tool') == 'flag_regulatory' and r.get('output')),
    None,
) or {}
_paths = [p['flag'] + '->' + p['severity'] + '->' + p['action']
          for p in _flag_out.get('severity_paths', [])]
print(f"GMS graph reached: {_paths or '(flag_regulatory was denied — no graph path recorded)'}")

## Run a PII case — caught at the first tool call

In [ ]:
case = next(c for c in cases if c['id'] == 'case-011')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
failed_tool = next(r for r in traj.records if r.action.kind == 'tool_call' and not r.observation.get('success'))
print('message:           ', case['message'])
print('failing gate:      ', failed_tool.observation['error'])

## Fact extraction as a policy-grounded query

It is tempting to read `extract_facts` as a classifier — turn the message into a `(product, issue)` label. Nothing downstream consumes that label for its own sake. Two tools consume the extraction: `search_policy` needs a *query* to retrieve the governing policy, and `flag_regulatory` needs a *signal* to ground its escalation. So the job is to turn the message into a **policy-grounded fact/query**.

`extract_facts` runs the policy store's parse-and-bind loop, binding the message to the real policy entities it implicates — the bound triples — and abstaining when nothing binds. That one artifact is the query the RAG retrieves on and the signal escalation grounds on. The product and issue are read off the bound entities. When the message binds to nothing, a Qwen-plus-rules fallback supplies the fields. `AGENTLAB_USE_LLM_EXTRACT=0` forces the pure-rule path (offline, no GPU).

In [ ]:
from forgeloop.agents.capstone.banking_tools import _extract_impl

for msg in [
    'I was charged a $35 overdraft fee on my checking account, reverse it!',
    'Someone made a charge on my account I never authorized.',
    "I'm going to sue you unless you give me my money back today.",   # case-009: vague
]:
    facts = _extract_impl(msg)
    print(msg)
    print('   product/issue:', facts['product'], '/', facts['issue'],
          '| query_facts:', facts['query_facts'], '| grounded:', bool(facts['extraction']))

A fee complaint binds to the `overdraft` / `fee_reversal` entities; an unauthorized-charge complaint binds to the dispute machinery; case-009 ("give me my money back") names no policy entity, so the grounding abstains and the fallback supplies a guarded `account_issue`. The bound `query_facts` are the workflow's pivot — `search_policy` reuses the same extraction (the message is parsed once) and `flag_regulatory` grounds its escalation signal on it:

```python
signal_from_query_facts([('overdraft', 'has_fee_amount', '?')])  # -> {'product': 'checking_account', 'issue': 'overdraft_fee'}
```

When grounding abstains, the fallback's two evidence guards forbid a regulatory issue the text does not support — `overdraft_fee` only stands with a fee/charge token, a `mortgage`/`loan` product only with a textual mention — so the model cannot manufacture a regulatory event from "give me my money back". Because nothing acts on the label itself, Chapter 17 grades this tool by downstream success — the retrieval and escalation its fact/query drives — not by a label match.

## Aggregate over all cases

Escalation accuracy is the headline. Classification accuracy is a secondary metric — when an early gate denies a tool call, the classifier never runs. That is correct governed behavior; you cannot classify a message you refuse to process.

In [ ]:
classify_correct = 0
classifier_ran   = 0
escalation_correct = 0
for case in cases:
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    out = traj.final_state.final_output or {}
    actual_class = out.get('classification') if isinstance(out, dict) else None
    if actual_class is None:
        for rec in traj.records:
            if rec.action.kind == 'tool_call' and rec.observation.get('success'):
                tool_out = rec.observation.get('output') or {}
                if isinstance(tool_out, dict) and 'category' in tool_out:
                    actual_class = tool_out['category']
                    break
    if actual_class is not None:
        classifier_ran += 1
    if actual_class == case.get('expected_classification'):
        classify_correct += 1
    if did_escalate(traj) == case.get('expected_escalation'):
        escalation_correct += 1

n = len(cases)
print(f'classification accuracy: {classify_correct}/{n} ({100*classify_correct/n:.0f}%)')
print(f'  of cases where the classifier ran: '
      f'{classify_correct}/{classifier_ran} ({100*classify_correct/classifier_ran:.0f}%)')
print(f'escalation accuracy:     {escalation_correct}/{n} ({100*escalation_correct/n:.0f}%)')
print(f'audit chain verifies:    {harness.audit.verify()}')

## Inspect the drafts

Complaint cases that classify as complaint and aren't escalated by `flag_regulatory` reach `draft_response`. The LoRA's output is short, on-template, and names the right policy with the right number.

In [ ]:
for case_id in ['case-003', 'case-005', 'case-020']:
    case = next(c for c in cases if c['id'] == case_id)
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    out = traj.final_state.final_output or {}
    print(f'[{case_id}] {case["message"][:55]!r}')
    print(f'  -> {out.get("draft_response", "")!r}')
    print()

Graph RAG's alias lexicon routes customer vocabulary (`chargeback`, `unauthorized transaction`, `fee waiver`) onto canonical policy ids. The LoRA was SFT-trained on the same canonical-summary template the kg-memory store now serves, so retrieval and generation are aligned on the same six policies. The split between LoRA-for-complaints and template-for-inquiry is intentional: the LoRA's training distribution is complaint→reply only.

## What the agent does and does not do

- It **classifies, extracts, retrieves policy, drafts**, and **escalates** when warranted.
- It does **not** invent facts, promise fee waivers, or process messages containing unredacted PII.
- Every step is recorded in a tamper-evident audit chain.

Where this would fall short in production:
- The policy library is small — six short text files plus one structured compendium. Real systems compose dozens, with versioning.
- The alias lexicon is hand-curated; customer vocabulary outside it (raw `SSN`, slang, regional spellings) still misses.
- The draft generator is a small LoRA adapter on a frozen Qwen3-4B, trained on a handful of templated policy replies — grounded but short and tied to the training template.
- The synthetic corpus is small. Real corpora are thousands; chunking and ranking start to matter.

In [ ]:
# Self-check
assert escalation_correct == n, f'expected 100% escalation accuracy, got {escalation_correct}/{n}'
assert harness.audit.verify()
print('OK')